## Vocabulary

Its a mapping between unique words and their integre index (which defines position of that word)

In [51]:
# let say we have a corpus of text
text = "In this paper, we show that the proposed approach of jointly learning to align and translate achieves significantly improved translation performance over the basic encoder–decoder approach. The im- provement is more apparent with longer sentences, but can be observed with sentences of any length. On the task of English-to-French translation, the proposed approach achieves, with a single model, a translation performance comparable, or close, to the conventional phrase-based system. Furthermore, qualitative analysis reveals that the proposed model finds a linguistically plausible (soft-)alignment between a source sentence and the corresponding target sentence."

In [54]:
# first of all I have to make a preprocessing pipeline
import string 
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize, sent_tokenize 

lemmatizer = WordNetLemmatizer()
puncts = set([punct for punct in string.punctuation])
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # convert to lowercase 
    text = text.lower()
    
    # tokenizing the text into sentences 
    sentences = sent_tokenize(text)
    sentences_words = [word_tokenize(sent) for sent in sentences]
    # now remove stop words
    result = []
    for i in range(len(sentences)):
        #sent = [word for word in sentences_words[i]] # split sentence into words
        rem_stp_word = [word for word in sentences_words[i] if word not in stop_words]
        rem_punct = [word for word in rem_stp_word if word not in puncts]
        lemmatized = [lemmatizer.lemmatize(w) for w in rem_punct]

        result.append(lemmatized)
    return result

In [55]:
text_new = preprocess_text(text)

for sent in text_new:
    print(sent)

['paper', 'show', 'proposed', 'approach', 'jointly', 'learning', 'align', 'translate', 'achieves', 'significantly', 'improved', 'translation', 'performance', 'basic', 'encoder–decoder', 'approach']
['im-', 'provement', 'apparent', 'longer', 'sentence', 'observed', 'sentence', 'length']
['task', 'english-to-french', 'translation', 'proposed', 'approach', 'achieves', 'single', 'model', 'translation', 'performance', 'comparable', 'close', 'conventional', 'phrase-based', 'system']
['furthermore', 'qualitative', 'analysis', 'reveals', 'proposed', 'model', 'find', 'linguistically', 'plausible', 'soft-', 'alignment', 'source', 'sentence', 'corresponding', 'target', 'sentence']


In [56]:
# Now we'll make vocabulary of all the unique words we have

word_set = set()

for sent in text_new:
    for word in sent:
        if word not in word_set:
            word_set.add(word)

word2idx = {word:idx for idx, word in enumerate(word_set)}
idx2word = {idx:word for idx, word in enumerate(word_set)}

print(f"Number of unique words we got : {len(word2idx)}")

Number of unique words we got : 43


In [58]:
# lets convert word to their idx

for sent in text_new:
    for i in range(len(sent)):
        sent[i] = word2idx[sent[i]]


for sent in text_new:
    print(sent)

[27, 17, 28, 37, 11, 18, 19, 10, 40, 32, 38, 13, 34, 20, 36, 37]
[29, 1, 14, 0, 21, 3, 21, 7]
[4, 33, 13, 28, 37, 40, 42, 9, 13, 34, 16, 22, 26, 41, 24]
[15, 5, 8, 6, 28, 9, 30, 23, 2, 35, 25, 12, 21, 31, 39, 21]


Here we got indexes to each word for the sentence, corresponding to vocabulary

## Some special tokens and padding

- &lt;UNK&gt; - unique token (if word is not present in vocabulary then it is assigned this token)
- &lt;SOS&gt; - start of sentence (indicates start of sentence)
- &lt;EOS&gt; - end of sentence (indicates end of sentence)

In [65]:
# Mostly what happens that each sentence don't have same number of words
# So to make each sentence of equal length we add pad tokens 
import torch

# Example sequences of varying lengths
sequences = []

for sent in text_new:
    tens = torch.tensor(sent)
    sequences.append(tens)

for sent in sequences:
    print(sent)


tensor([27, 17, 28, 37, 11, 18, 19, 10, 40, 32, 38, 13, 34, 20, 36, 37])
tensor([29,  1, 14,  0, 21,  3, 21,  7])
tensor([ 4, 33, 13, 28, 37, 40, 42,  9, 13, 34, 16, 22, 26, 41, 24])
tensor([15,  5,  8,  6, 28,  9, 30, 23,  2, 35, 25, 12, 21, 31, 39, 21])


In [64]:
# you can see each sentence is not of equal length
# we'll find sentence which have max length
max_len = 0

for sent in sequences:
    sent_len = len(sent)
    if sent_len > max_len:
        max_len = sent_len 

print(max_len)

16


In [66]:
# Pad the sequences
from torch.nn.utils.rnn import pad_sequence
padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=456)

print(padded_sequences)

tensor([[ 27,  17,  28,  37,  11,  18,  19,  10,  40,  32,  38,  13,  34,  20,
          36,  37],
        [ 29,   1,  14,   0,  21,   3,  21,   7, 456, 456, 456, 456, 456, 456,
         456, 456],
        [  4,  33,  13,  28,  37,  40,  42,   9,  13,  34,  16,  22,  26,  41,
          24, 456],
        [ 15,   5,   8,   6,  28,   9,  30,  23,   2,  35,  25,  12,  21,  31,
          39,  21]])


we can see that empty places in shorter sentences are filled with pading token